In [17]:
# MAIN PARAMETERS
ACTIVE_MODEL = "linear_regression"

# linear_regression, xgboost

NUMERIC_SKEWED_FEATURES = [
    "sales_lag15",
    "sales_lag16",
    "sales_lag17",
    "sales_lag18",
    "sales_lag19",
    "sales_lag20",
    "sales_lag21",
    "sales_lag30",
    "sales_lag60",
    "sales_lag364",
    "sales_lag365",
]
NUMERIC_NORMAL_FEATURES = []

TIER_FEATURES = []
CATEGORICAL_FEATURES = []
TARGET_ENCODER = False

TARGET = "sales"

COLUMNS_TO_KEEP = ["date", "family", "store_nbr", "is_int_sales"]

CUTOFFS_STR = ["2017-07-15", "2017-04-15", "2017-01-15", "2016-09-15"]
CUTOFFS_DATETIME = pd.to_datetime(CUTOFFS_STR)

HORIZON = 15

ISSUE_NAME = "lags"
VERSION_NAME = "[15-21]+30,60,364,365"

PATH_TRAIN_DF = "../data/processed/df_train.parquet"

In [21]:
import statsmodels.api as sm
import numpy as np
import pandas as pd

df = pd.read_parquet(PATH_TRAIN_DF)

cutoff_date = max(CUTOFFS_STR)
df_train = df[df["date"] <= cutoff_date].copy()

y_train_eda = np.log1p(df_train["sales"])
X_train_eda = df_train[NUMERIC_SKEWED_FEATURES].copy()

for col in NUMERIC_SKEWED_FEATURES:
    X_train_eda[col] = np.log1p(X_train_eda[col])

X_train_sm = sm.add_constant(X_train_eda)
model_sm = sm.OLS(y_train_eda, X_train_sm).fit()

print(model_sm.summary2())

                   Results: Ordinary least squares
Model:              OLS              Adj. R-squared:     0.904       
Dependent Variable: sales            AIC:                5677707.6423
Date:               2026-11-16 13:28 BIC:                5677859.4741
No. Observations:   2309736          Log-Likelihood:     -2.8388e+06 
Df Model:           11               F-statistic:        1.988e+06   
Df Residuals:       2309724          Prob (F-statistic): 0.00        
R-squared:          0.904            Scale:              0.68405     
----------------------------------------------------------------------
                  Coef.   Std.Err.     t      P>|t|    [0.025   0.975]
----------------------------------------------------------------------
const             0.1138    0.0008  134.3666  0.0000   0.1121   0.1155
sales_lag15       0.2238    0.0008  265.5329  0.0000   0.2221   0.2255
sales_lag16       0.1423    0.0009  160.5028  0.0000   0.1406   0.1441
sales_lag17       0.0808    0.000

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["Variable"] = X_train_sm.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_train_sm.values, i) for i in range(X_train_sm.shape[1])
]

print(vif_data.sort_values("VIF", ascending=False))

        Variable        VIF
4    sales_lag18  19.159414
2    sales_lag16  19.071418
5    sales_lag19  18.436788
3    sales_lag17  18.419123
6    sales_lag20  18.197335
7    sales_lag21  17.717659
1    sales_lag15  17.221930
10  sales_lag364  14.586440
11  sales_lag365  14.501723
8    sales_lag30  12.573067
9    sales_lag60   7.835885
0          const   1.000000
